In [9]:
# =====================================
# 1. IMPORT REQUIRED LIBRARIES
# =====================================
import pandas as pd
import joblib
import os


# =====================================
# 2. LOAD DATASET (CSV → PKL SAFE)
# =====================================
CSV_PATH = "data/doctors.csv"
PKL_PATH = "models/doctors.pkl"

# Load from CSV if exists, else from PKL
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("✅ Loaded doctors from CSV")
else:
    df = joblib.load(PKL_PATH)
    print("✅ Loaded doctors from PKL")


# =====================================
# 3. STANDARDIZE COLUMN NAMES
# =====================================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("📌 Columns after standardization:")
print(df.columns.tolist())


# =====================================
# 4. FIX BROKEN CONSULTATION COLUMN
# =====================================
# Handles cases like:
# emailconsultation, consultationindex, etc.
for col in df.columns:
    if "consultation" in col and col != "consultation":
        df.rename(columns={col: "consultation"}, inplace=True)

# Safety check
if "consultation" not in df.columns:
    raise ValueError("❌ consultation column missing in dataset")

print("✅ Consultation column verified")


# =====================================
# 5. CREATE doctor_name COLUMN
# =====================================
df["doctor_name"] = df["first_name"] + " " + df["last_name"]


# =====================================
# 6. STANDARDIZE EXPERIENCE COLUMN
# =====================================
if "years_experience" in df.columns:
    df.rename(columns={"years_experience": "experience"}, inplace=True)

df["experience"] = df["experience"].astype(int)


# =====================================
# 7. DISEASE → SPECIALIZATION MAP
# =====================================
disease_specialization_map = {
    "diabetes": ["Endocrinology"],
    "heart": ["Cardiology"],
    "liver": ["Gastroenterology", "Hepatology"],
    "breast_cancer": ["Oncology"]
}


# =====================================
# 8. DOCTOR RECOMMENDATION FUNCTION
# (TOP 3 DOCTORS)
# =====================================
def recommend_doctors(disease, top_n=3):
    """
    Returns top N doctors based on:
    - Disease
    - Specialization
    - Experience (DESC)
    """

    if disease not in disease_specialization_map:
        return pd.DataFrame()

    specs = disease_specialization_map[disease]

    filtered_df = df[df["specialization"].isin(specs)]

    if filtered_df.empty:
        return pd.DataFrame()

    top_doctors = (
        filtered_df
        .sort_values(by="experience", ascending=False)
        .head(top_n)
    )

    return top_doctors[[
        "doctor_id",
        "doctor_name",
        "specialization",
        "experience",
        "hospital_branch",
        "phone_number",
        "email",
        "consultation"
    ]]


# =====================================
# 9. TEST OUTPUT
# =====================================
print("\n🩺 Top Doctors for Diabetes:")
print(recommend_doctors("diabetes"))

print("\n❤️ Top Doctors for Heart Disease:")
print(recommend_doctors("heart"))

print("\n🫁 Top Doctors for Liver Disease:")
print(recommend_doctors("liver"))

print("\n🎗 Top Doctors for Breast Cancer:")
print(recommend_doctors("breast_cancer"))


# =====================================
# 10. FRONTEND SAFE RESPONSE
# =====================================
response = recommend_doctors("liver")

print("\n📤 API Response (Frontend Ready):")
print(response.to_dict(orient="records"))


# =====================================
# 11. SAVE CLEAN PKL
# =====================================
os.makedirs("models", exist_ok=True)
joblib.dump(df, PKL_PATH)

print("\n✅ Clean doctors.pkl saved successfully")


✅ Loaded doctors from CSV
📌 Columns after standardization:
['doctor_id', 'first_name', 'last_name', 'specialization', 'phone_number', 'years_experience', 'hospital_branch', 'email', 'consultation']
✅ Consultation column verified

🩺 Top Doctors for Diabetes:
   doctor_id    doctor_name specialization  experience   hospital_branch  \
12      D013    Priya Mehta  Endocrinology          18   Eastside Clinic   
18      D019  Sunita Kapoor  Endocrinology          16   Westside Clinic   
13      D014     Neha Gupta  Endocrinology          12  Central Hospital   

    phone_number                          email  consultation  
12    9988776655    dr.priya.mehta@hospital.com          1600  
18    9321456789  dr.sunita.kapoor@hospital.com          1600  
13    9090909090     dr.neha.gupta@hospital.com          1600  

❤️ Top Doctors for Heart Disease:
   doctor_id   doctor_name specialization  experience   hospital_branch  \
11      D012    Amit Verma     Cardiology          22   Westside Clinic

In [6]:
# ADD consultation_fee column (IMPORTANT)
df["consultation_fee"] = 500  # default fee
